# Notebook 03 — Model Training & Evaluation

**Goal:** Train Logistic Regression, Random Forest, and XGBoost on the engineered features.
Evaluate each model on the held-out test set. Save the best model and metrics.

**Metric note:** Two F1 scores are reported:
- `f1` — binary F1 for the **churn class only** (minority class; harder to optimise)
- `f1_weighted` — weighted average F1 across both classes, accounting for class imbalance

The ~0.79 F1 figure commonly cited for this dataset refers to the **weighted F1**.

In [ ]:
import sys
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.train_models import split_data, train_logistic_regression, train_random_forest, train_xgboost, save_model, find_optimal_threshold
from src.evaluate import compute_metrics, plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve, plot_feature_importance, save_metrics

sns.set_theme(style='whitegrid', palette='muted')
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Ready.')

## 1. Load Engineered Features

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'processed' / 'telco_churn_features.csv')
print('Shape:', df.shape)
print('Churn distribution:')
print(df['Churn'].value_counts())

## 2. Train / Test Split (Stratified 80/20)

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean()*100:.1f}%  |  Test churn rate: {y_test.mean()*100:.1f}%')

## 3. Logistic Regression (Baseline)

Balanced class weights + L2 regularisation. Decision threshold is tuned to maximise F1 on training data.

In [ ]:
lr = train_logistic_regression(X_train, y_train)

lr_prob = lr.predict_proba(X_test)[:, 1]
lr_thr  = find_optimal_threshold(lr, X_train, y_train)
lr_pred = (lr_prob >= lr_thr).astype(int)
lr_metrics = compute_metrics(y_test, lr_pred, lr_prob)

print(f'Optimal threshold: {lr_thr:.3f}')
print('Logistic Regression Metrics:')
for k, v in lr_metrics.items():
    print(f'  {k:15s}: {v}')

In [ ]:
_ = plot_confusion_matrix(y_test, lr_pred, 'Logistic Regression')
_ = plot_roc_curve(y_test, lr_prob, 'Logistic Regression')
_ = plot_precision_recall_curve(y_test, lr_prob, 'Logistic Regression')

## 4. Random Forest (Tuned)

GridSearchCV over max_depth and min_samples_leaf. 200 estimators, balanced class weights.

In [ ]:
rf = train_random_forest(X_train, y_train)

rf_prob = rf.predict_proba(X_test)[:, 1]
rf_thr  = find_optimal_threshold(rf, X_train, y_train)
rf_pred = (rf_prob >= rf_thr).astype(int)
rf_metrics = compute_metrics(y_test, rf_pred, rf_prob)

print(f'Optimal threshold: {rf_thr:.3f}')
print('Random Forest Metrics:')
for k, v in rf_metrics.items():
    print(f'  {k:15s}: {v}')

In [ ]:
_ = plot_confusion_matrix(y_test, rf_pred, 'Random Forest')
_ = plot_roc_curve(y_test, rf_prob, 'Random Forest')
_ = plot_precision_recall_curve(y_test, rf_prob, 'Random Forest')
_ = plot_feature_importance(rf, X_train.columns.tolist(), 'Random Forest')

## 5. XGBoost (Best Model)

GridSearchCV over learning_rate, max_depth, n_estimators, subsample, min_child_weight.
Trained on raw class distribution; threshold optimised for maximum F1.

In [ ]:
xgb, xgb_thr = train_xgboost(X_train, y_train)

xgb_prob = xgb.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_prob >= xgb_thr).astype(int)
xgb_metrics = compute_metrics(y_test, xgb_pred, xgb_prob)

print(f'Optimal threshold: {xgb_thr:.3f}')
print('XGBoost Metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k:15s}: {v}')

In [ ]:
_ = plot_confusion_matrix(y_test, xgb_pred, 'XGBoost')
_ = plot_roc_curve(y_test, xgb_prob, 'XGBoost')
_ = plot_precision_recall_curve(y_test, xgb_prob, 'XGBoost')
_ = plot_feature_importance(xgb, X_train.columns.tolist(), 'XGBoost')

## 6. Model Comparison Summary

In [ ]:
summary = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Random Forest':       rf_metrics,
    'XGBoost':             xgb_metrics,
}).T

print('=== MODEL COMPARISON ===')
print(summary.to_string())
summary

In [ ]:
# Multi-model ROC curves
fig, ax = plt.subplots(figsize=(7, 6))
for name, prob in [('Logistic Regression', lr_prob), ('Random Forest', rf_prob), ('XGBoost', xgb_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc_val:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'roc_curves_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Best Model & Metrics

In [ ]:
save_model(xgb, 'best_model')

all_metrics = {
    'Logistic Regression': lr_metrics,
    'Random Forest':       rf_metrics,
    'XGBoost':             xgb_metrics,
}
save_metrics(all_metrics)
print('Saved.')

## 8. Generate Tableau Export with Predicted Probabilities

In [ ]:
import joblib
from src.feature_engineering import build_features

df_clean = pd.read_csv(ROOT / 'data' / 'processed' / 'telco_churn_cleaned.csv')
df_full, _ = build_features(df_clean.copy())
X_full = df_full.drop(columns=['Churn']).reindex(columns=X_train.columns, fill_value=0)
pred_probs = xgb.predict_proba(X_full)[:, 1]

bins   = [0, 12, 24, 48, df_clean['tenure'].max()+1]
labels = ['0-12','13-24','25-48','49+']
df_clean['tenure_group']               = pd.cut(df_clean['tenure'], bins=bins, labels=labels).astype(str)
df_clean['clv']                        = df_clean['MonthlyCharges'] * df_clean['tenure']
df_clean['predicted_churn_probability'] = pred_probs
df_clean['risk_tier'] = pd.cut(pred_probs, bins=[0, 0.35, 0.65, 1.0], labels=['Low','Medium','High']).astype(str)

tableau_path = ROOT / 'data' / 'tableau' / 'churn_dashboard_data.csv'
tableau_path.parent.mkdir(exist_ok=True)
df_clean.to_csv(tableau_path, index=False)
print(f'Tableau export saved: {tableau_path}')
print(f'Shape: {df_clean.shape}')
print('Risk tier distribution:')
print(df_clean['risk_tier'].value_counts())